# 03 — Data Cleaning and Preprocessing (Stage 3)

**Idea:** label every row with what it is, then decide what to use. Nothing is silently dropped; every exclusion is counted.

| TransactionType | Meaning |
|---|---|
| SALE | Real fulfilled product sale → used for demand and revenue |
| REVERSED_SALE | Sale cancelled in full later (same customer, product, quantity) → never fulfilled |
| RETURN | Product line on a 'C' (cancellation) invoice |
| CANCELLATION_NON_PRODUCT | 'C' line for discounts, postage refunds, fees |
| STOCK_ADJUSTMENT | Negative quantity, zero price, normal invoice (damaged / lost) |
| ACCOUNTING_ADJUSTMENT | 'A' invoices (bad debt) |
| NON_PRODUCT | Postage, carriage, bank charges, Amazon fees, manual lines, gift vouchers |
| ZERO_PRICE | Positive quantity but price 0 |
| INVALID | Missing date / code / quantity |

**Forecast target:** gross fulfilled demand = units of SALE rows per product per day. Returns are tracked separately, not subtracted.

> **Colab:** run the setup cell; it asks for the project ZIP the first time. Put `Online Retail.xlsx` in `data/raw/` (or run notebook 01 first, which uploads it).

In [1]:
import sys, os, zipfile, io
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd()
for c in (ROOT, ROOT.parent, ROOT / "AI-Sales-Forecasting-Inventory"):
    if (c / "ml").is_dir():
        ROOT = c
        break
if IN_COLAB and not (ROOT / "ml").is_dir():
    from google.colab import files
    print("Upload the project ZIP once (AI-Sales-Forecasting-Inventory.zip):")
    for name, data in files.upload().items():
        zipfile.ZipFile(io.BytesIO(data)).extractall("/content")
    ROOT = Path("/content/AI-Sales-Forecasting-Inventory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["SFIO_PROJECT_ROOT"] = str(ROOT)

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Image, display, Markdown
from ml import config
config.ensure_directories()
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
print("Project root:", ROOT.name, "| Colab:", IN_COLAB)

Project root: AI-Sales-Forecasting-Inventory | Colab: False


In [2]:
from ml.data_loader import load_dataset
from ml import preprocessing
raw = load_dataset().df
result = preprocessing.preprocess(raw)
c = result.report["counts"]
pd.Series({k: v for k, v in c.items() if isinstance(v, int)}, name="rows").to_frame()

,rows
original_rows,541909
missing_value_cells_total,136534
duplicate_rows_removed,5268
rows_after_deduplication,536641
invalid_rows,0
cancellation_lines,9251
cancelled_invoices,3836
return_lines,8668
reversed_sale_lines,2805
stock_adjustment_lines,1336


## 1. Rows per transaction type and their value

In [3]:
pd.DataFrame(result.report["transaction_types"]).T[["rows", "revenue", "meaning"]]

,rows,revenue,meaning
SALE,519735,9823459.37,Real fulfilled product sale (positive quantity...
REVERSED_SALE,2805,423759.95,Sale cancelled in full later by the same custo...
RETURN,8668,-475901.16,Product line on a cancellation ('C') invoice: ...
CANCELLATION_NON_PRODUCT,583,-418078.57,"Cancellation line for discounts, postage, fees..."
STOCK_ADJUSTMENT,1336,0.0,Negative quantity with zero price on a normal ...
ACCOUNTING_ADJUSTMENT,3,-11062.06,Bad-debt adjustment invoices starting with 'A'...
NON_PRODUCT,2354,383829.42,"Postage, carriage, bank charges, marketplace f..."
ZERO_PRICE,1157,0.0,"Positive quantity with zero price: no revenue,..."
INVALID,0,0.0,"Missing/invalid timestamp, StockCode or quantity."


## 2. Cancellations are not blindly deleted
A cancellation that exactly reverses an earlier sale (same customer, product and quantity) marks that sale as REVERSED_SALE. Partial returns leave the original sale untouched.

In [4]:
rev = result.classified[result.classified["TransactionType"] == "REVERSED_SALE"]
rev.nlargest(5, "Quantity")[["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "CustomerID"]]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,CustomerID
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,16446
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,12346
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,15749
52710,540815,21175,GIN + TONIC DIET METAL SIGN,2000,2011-01-11 12:55:00,15749
52709,540815,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1930,2011-01-11 12:55:00,15749


## 3. Outliers: investigated, not deleted
The famous 80,995- and 74,215-unit lines were cancelled minutes later, so the REVERSED_SALE rule removes them for a real reason. Other big lines are wholesale bulk orders: **flagged and kept**.

In [5]:
display(pd.DataFrame(result.report["outliers"]["largest_flagged_lines"]))
print(result.report["outliers"]["rule"])

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,InvoiceDate,CustomerID
0,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,0.21,2011-10-27T12:26:00,12901
1,554868,22197,POPCORN HOLDER,4300,0.72,2011-05-27T10:52:00,13135
2,544612,22053,EMPIRE DESIGN ROSETTE,3906,0.82,2011-02-22T10:43:00,18087
3,560599,18007,ESSENTIAL BALM 3.5G TIN IN ENVELOPE,3186,0.06,2011-07-19T17:04:00,14609
4,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,2011-04-18T13:20:00,15749
5,573995,16014,SMALL CHINESE STYLE SCISSOR,3000,0.32,2011-11-02T11:24:00,16308
6,536830,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,2880,0.18,2010-12-02T17:38:00,16754
7,562439,84879,ASSORTED COLOUR BIRD ORNAMENT,2880,1.45,2011-08-04T18:06:00,12931
8,554272,21977,PACK OF 60 PINK PAISLEY CAKE CASES,2700,0.42,2011-05-23T13:08:00,12901
9,543057,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,2592,0.21,2011-02-03T10:50:00,16333


Flag if quantity > product's 99th percentile AND >= 10x product median AND >= 100 units. Flagged rows are KEPT.


## 4. Inconsistent descriptions
One canonical description per StockCode (most frequent among sales), upper-cased and trimmed.

In [6]:
cl = result.classified
changed = cl[cl["DescriptionRaw"].fillna("").str.upper().str.strip() != cl["Description"].fillna("")]
changed[["StockCode", "DescriptionRaw", "Description"]].drop_duplicates().head(10)

,StockCode,DescriptionRaw,Description
8,22632,HAND WARMER RED POLKA DOT,HAND WARMER RED RETROSPOT
87,85099C,JUMBO BAG BAROQUE BLACK WHITE,JUMBO BAG BAROQUE BLACK WHITE
90,84997B,RED 3 PIECE RETROSPOT CUTLERY SET,CHILDRENS CUTLERY RETROSPOT RED
91,84997C,BLUE 3 PIECE POLKADOT CUTLERY SET,CHILDRENS CUTLERY POLKADOT BLUE
128,84832,ZINC WILLIE WINKIE CANDLE STICK,ZINC WILLIE WINKIE CANDLE STICK
154,35004C,SET OF 3 COLOURED FLYING DUCKS,SET OF 3 COLOURED FLYING DUCKS
181,22466,FAIRY TALE COTTAGE NIGHTLIGHT,FAIRY TALE COTTAGE NIGHT LIGHT
187,21411,GINGHAM HEART DOORSTOP RED,GINGHAM HEART DOORSTOP RED
190,22915,ASSORTED BOTTLE TOP MAGNETS,ASSORTED BOTTLE TOP MAGNETS
221,22197,SMALL POPCORN HOLDER,POPCORN HOLDER


## 5. Revenue definitions (reported separately)

In [7]:
pd.Series(result.report["revenue"]).to_frame("value")

,value
gross_fulfilled_sales_revenue,9823459.37
gross_invoiced_revenue_incl_reversed,10247219.32
cancellations_and_returns_value,-893979.73
net_revenue_after_cancellations,9353239.59
note,Dashboard 'revenue' = gross fulfilled sales re...


## 6. Save outputs
`cleaned_sales.csv/.parquet`, `transactions_classified.parquet`, `product_catalog.parquet`, `03_preprocessing_report.json/.md`.

In [8]:
preprocessing.save_outputs(result)
print("Final cleaned dataset:", result.report["counts"]["final_dimensions"])
result.sales.head()

Final cleaned dataset: {'rows': 519735, 'columns': 11}


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Date,IsExtremeQuantity
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12-01,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12-01,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01,False
